# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TEJAS7356/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content item (page) for one client on one report date.

For this assignment, I will work with the daily content-performance data for a mid-panel month, March 2026. The analysis uses the warehouse daily performance table, fact_content_daily_performance.

The decision unit is therefore client × content item × day. The time window used for the initial verification is 2026-03-01 to 2026-03-31.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

FEATURES
- impressions_90d — recent search visibility available before the decision.
- clicks_90d — recent search traffic available before the decision.
- sessions_90d — recent site activity available before the decision.
- content_age_days — age of the content available at the decision moment.
- days_since_last_update — content freshness available at the decision moment.

LABEL
- trend_direction — used to define the declining outcome.
- is_declining_label — target: 1 when trend_direction is "down", otherwise 0.

CONTEXT
- content_id — used to identify and group content items, not as a model feature.
- client_id — used for client grouping and holdout splits, not as a model feature.
- content_type — describes the type of content.
- main_intent — describes the search intent context.
- competition_level — describes keyword competition context.

EXCLUDED
- trend_pct — excluded because it is derived from the same trend information used to create the label and would cause leakage.
- provider_used — excluded because the data dictionary says it is not a model feature.
- model_used — excluded because the data dictionary says it is not a model feature.
- avg_position — excluded from this small feature set because I am limiting the initial feature frame to five features.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"
DIM = f"read_parquet('{REL}/dim_content.parquet')"
CLI = f"read_parquet('{REL}/dim_clients.parquet')"

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

print("Connected successfully.")
print("Feature window: February 2026")
print("Label window: March 2026")

# Query 1 — Verify the grain
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT report_date || '|' || client_hash_id || '|' || content_hash_id)
        AS distinct_grain_rows
FROM {MAR}
""")

Connected successfully.
Feature window: February 2026
Label window: March 2026


┌────────────┬─────────────────────┐
│ total_rows │ distinct_grain_rows │
│   int64    │        int64        │
├────────────┼─────────────────────┤
│    9841378 │             9841378 │
└────────────┴─────────────────────┘

In [12]:
# Query 2 — Row count and date span
con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM {FEB}
""")

┌───────────┬────────────┬────────────┐
│ row_count │ first_date │ last_date  │
│   int64   │    date    │    date    │
├───────────┼────────────┼────────────┤
│   7355108 │ 2026-02-01 │ 2026-02-28 │
└───────────┴────────────┴────────────┘

In [13]:
# Query 3 — Availability
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS available_rows
FROM {FEB}
""")

┌────────────┬────────────────┐
│ total_rows │ available_rows │
│   int64    │     int64      │
├────────────┼────────────────┤
│    7355108 │        2621783 │
└────────────┴────────────────┘

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This data cannot tell us the true cause of changes in search performance. The available history may be unbalanced across content and time periods, and some early rows may contain GSC-only measurements. Feature and label windows can also overlap, which limits causal interpretation. Therefore, the results should be treated as observed, directional decision-support signals rather than proof of causation.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.